# 01 — Annotation naive

This notebook is the naive model-screening experiment. It sends the same
full-codebook prompt (`P0`) and the same ten validation dialogues to every
model. No model-specific prompt tuning is used, so the comparison tests how
well each model follows the annotation task natively.

Results are cached under:

```text
extension/artifacts/extraction_cache/naive_testing/{model}/P0/{dialogue_id}.json
```

The extraction cell is disabled by default. The results cell reads the
existing cache and does not call an API.


## 1. Load the validation dataset

The human validation set supplies the dialogue text sent to the models and
the gold annotations used for scoring. The prompt receives only the
conversation and unit names; gold labels and adjudication fields remain
hidden.


In [9]:
import os
import sys
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / 'extension' / 'artifacts').exists():
        os.chdir(candidate)
        break
else:
    raise FileNotFoundError('Run this notebook from inside the Experiment_1 repository.')

sys.path.insert(0, str(Path.cwd()))

if not os.environ.get('OPENROUTER_API_KEY') and Path('.env').exists():
    for line in Path('.env').read_text().splitlines():
        if line.strip().startswith('OPENROUTER_API_KEY='):
            os.environ['OPENROUTER_API_KEY'] = line.split('=', 1)[1].strip().strip('\"').strip("'")
            break

print('repository:', Path.cwd())
print('OPENROUTER_API_KEY set:', bool(os.environ.get('OPENROUTER_API_KEY')))


repository: /Users/tandon.utsav2/Desktop/Experiment_1
OPENROUTER_API_KEY set: True


In [10]:
from extension.scripts.load_annotation_data import load_dataset
from extension.scripts import extraction, prompt_loader, scoring

VALIDATION_PATH = Path('extension/artifacts/annotation_dev_and_val_sets/validation_set.csv')
CACHE_NAMESPACE = 'naive_testing'
PROMPT = 'P0'
N_DIALOGUES = 10

gold = load_dataset(VALIDATION_PATH)
dialogues = extraction.dialogues_from(gold, split=CACHE_NAMESPACE)
test_dialogues = dialogues[:N_DIALOGUES]
dialogue_ids = [dialogue['dialogue_id'] for dialogue in test_dialogues]

assert PROMPT in prompt_loader.list_prompts(), prompt_loader.list_prompts()
assert len(test_dialogues) == N_DIALOGUES
print(f'loaded {len(gold)} annotated units across {len(dialogues)} dialogues')
print('test dialogue ids:', dialogue_ids)
print('cache root:', Path('extension/artifacts/extraction_cache') / CACHE_NAMESPACE)


loaded 544 annotated units across 78 dialogues
test dialogue ids: [1, 21, 35, 79, 143, 178, 255, 270, 275, 289]
cache root: extension/artifacts/extraction_cache/naive_testing


## 2. Model configuration

All models are held in one configuration list. `reasoning_effort` is used
when OpenRouter exposes an effort selector; `reasoning_enabled=True` is used
when the provider supports reasoning but manages its own reasoning budget.
Models without either control receive no reasoning field. Temperature is
zero for every request.


In [11]:
import pandas as pd

MODELS = [
    {'group': 'large', 'name': 'GLM 5.2', 'model': 'z-ai/glm-5.2', 'reasoning_effort': 'xhigh', 'reasoning_enabled': None, 'reasoning_setting': 'xhigh'},
    {'group': 'large', 'name': 'Kimi K3', 'model': 'moonshotai/kimi-k3', 'reasoning_effort': 'max', 'reasoning_enabled': None, 'reasoning_setting': 'max'},
    {'group': 'large', 'name': 'Qwen3.8 Max', 'model': 'qwen/qwen3.8-max', 'reasoning_effort': 'xhigh', 'reasoning_enabled': None, 'reasoning_setting': 'xhigh'},
    {'group': 'medium', 'name': 'Qwen3.5 122B-A10B', 'model': 'qwen/qwen3.5-122b-a10b', 'reasoning_effort': None, 'reasoning_enabled': True, 'reasoning_setting': 'enabled (model-managed)'},
    {'group': 'medium', 'name': 'GPT-OSS 120B', 'model': 'openai/gpt-oss-120b', 'reasoning_effort': 'high', 'reasoning_enabled': None, 'reasoning_setting': 'high'},
    {'group': 'medium', 'name': 'Nemotron 3 Super 120B-A12B', 'model': 'nvidia/nemotron-3-super-120b-a12b', 'reasoning_effort': 'medium', 'reasoning_enabled': None, 'reasoning_setting': 'medium'},
    {'group': 'small', 'name': 'Qwen3.5 35B-A3B', 'model': 'qwen/qwen3.5-35b-a3b', 'reasoning_effort': None, 'reasoning_enabled': True, 'reasoning_setting': 'enabled (model-managed)'},
    {'group': 'small', 'name': 'Qwen3.5 27B', 'model': 'qwen/qwen3.5-27b', 'reasoning_effort': None, 'reasoning_enabled': True, 'reasoning_setting': 'enabled (model-managed)'},
    {'group': 'small', 'name': 'GPT-OSS 20B', 'model': 'openai/gpt-oss-20b', 'reasoning_effort': 'high', 'reasoning_enabled': None, 'reasoning_setting': 'high'},
    {'group': 'small', 'name': 'Gemma 3 12B', 'model': 'google/gemma-3-12b-it', 'reasoning_effort': None, 'reasoning_enabled': None, 'reasoning_setting': 'unsupported'},
    {'group': 'small', 'name': 'Qwen3.5 9B', 'model': 'qwen/qwen3.5-9b', 'reasoning_effort': None, 'reasoning_enabled': True, 'reasoning_setting': 'enabled (model-managed)'},
    {'group': 'small', 'name': 'Gemma 3 4B', 'model': 'google/gemma-3-4b-it', 'reasoning_effort': None, 'reasoning_enabled': None, 'reasoning_setting': 'unsupported'},
]

MAX_WORKERS = 2
extraction.TEMPERATURE = 0.0
pd.DataFrame(MODELS)[['group', 'name', 'model', 'reasoning_setting']]


,group,name,model,reasoning_setting
0,large,GLM 5.2,z-ai/glm-5.2,xhigh
1,large,Kimi K3,moonshotai/kimi-k3,max
2,large,Qwen3.8 Max,qwen/qwen3.8-max,xhigh
3,medium,Qwen3.5 122B-A10B,qwen/qwen3.5-122b-a10b,enabled (model-managed)
4,medium,GPT-OSS 120B,openai/gpt-oss-120b,high
5,medium,Nemotron 3 Super 120B-A12B,nvidia/nemotron-3-super-120b-a12b,medium
6,small,Qwen3.5 35B-A3B,qwen/qwen3.5-35b-a3b,enabled (model-managed)
7,small,Qwen3.5 27B,qwen/qwen3.5-27b,enabled (model-managed)
8,small,GPT-OSS 20B,openai/gpt-oss-20b,high
9,small,Gemma 3 12B,google/gemma-3-12b-it,unsupported


## 3. Extraction

Leave `SKIP = True` when reviewing cached results. Set it to `False` only
when inference is intended. The shared extraction script reuses valid cache
records and requests only missing or invalid records. Consequently, changing
`SKIP` can incur API costs and will retry model-limitation failures as well as
transport failures.


In [12]:
SKIP = True

if SKIP:
    print('Extraction skipped; existing cache files are unchanged.')
else:
    if not os.environ.get('OPENROUTER_API_KEY'):
        raise RuntimeError('Set OPENROUTER_API_KEY before running extraction.')

    for config in MODELS:
        print(
            f"{config['group'].upper()} | {config['name']} | "
            f"reasoning={config['reasoning_setting']}"
        )
        extraction.generate_annotations(
            PROMPT,
            config['model'],
            test_dialogues,
            max_workers=MAX_WORKERS,
            reasoning_effort=config['reasoning_effort'],
            reasoning_enabled=config['reasoning_enabled'],
        )


Extraction skipped; existing cache files are unchanged.


## 4. Cached results

The four reported metrics are calculated by the shared scoring script.
`valid_rate` uses all ten requested dialogues. The remaining metrics use only
valid annotations, so results based on a low valid rate are vulnerable to
selection bias and should not be compared directly with complete results.


In [13]:
score_rows = []
for config in MODELS:
    metrics = scoring.score_config(
        gold,
        config['model'],
        PROMPT,
        dialogue_ids,
        n_boot=0,
        split=CACHE_NAMESPACE,
    )
    score_rows.append({
        'group': config['group'],
        'model': config['name'],
        'valid_rate': metrics['valid_rate'],
        'macro_f1': metrics['macro_f1_P'],
        'accuracy': metrics['accuracy'],
        'alpha': metrics['alpha'],
    })

results = pd.DataFrame(score_rows)
display(results.round(3))


,group,model,valid_rate,macro_f1,accuracy,alpha
0,large,GLM 5.2,0.6,0.256,0.818,0.564
1,large,Kimi K3,1.0,0.623,0.926,0.826
2,large,Qwen3.8 Max,1.0,0.485,0.789,0.455
3,medium,Qwen3.5 122B-A10B,0.5,0.468,0.749,0.305
4,medium,GPT-OSS 120B,1.0,0.361,0.633,0.327
5,medium,Nemotron 3 Super 120B-A12B,0.6,0.156,0.615,0.207
6,small,Qwen3.5 35B-A3B,0.2,0.722,0.757,0.421
7,small,Qwen3.5 27B,0.7,0.423,0.790,0.433
8,small,GPT-OSS 20B,0.8,0.090,0.569,0.076
9,small,Gemma 3 12B,0.8,0.048,0.317,-0.202


## 5. Failure analysis and model choice

The cached run shows two distinct failure layers. A **format failure** means
the response could not pass the annotation schema and therefore contributes
only to `valid_rate`. A **semantic failure** means the response was valid JSON
but its labels disagreed with the human annotation.

| Model | Valid | Principal failure mode | Interpretation |
|---|---:|---|---|
| GLM 5.2 | 6/10 | Four responses spent their output on reasoning and never produced annotation JSON; valid responses also missed relevance and most wrong-operation errors. | Capable reasoning did not translate reliably into a final structured answer. |
| **Kimi K3** | **10/10** | No format failures. Its main semantic weakness was under-detecting wrong-operation errors and occasionally predicting principles where gold had none. | It followed the long codebook and output contract consistently while preserving strong label agreement. |
| Qwen3.8 Max | 10/10 | No format failures, but it overused `N`, missed many relevance errors, and produced many false wrong-operation positives. | Reliable serialization alone was insufficient; its interpretation of family boundaries was weaker. |
| Qwen3.5 122B-A10B | 5/10 | Two reasoning-only responses, two malformed JSON responses, and one resolution-invariant failure; valid outputs rarely used `A` and missed relevance. | Both completion reliability and label calibration were unstable. |
| GPT-OSS 120B | 10/10 | No format failures, but it heavily overpredicted `A`, under-detected wrong-operation errors, and invented principles/steps evidence. | High compliance, but weak calibration to the codebook's engagement and family boundaries. |
| Nemotron 3 Super 120B-A12B | 6/10 | Four reasoning-only responses; valid outputs missed relevance and wrong-operation errors and overcalled comprehension. | Long reasoning frequently displaced the required final answer, and the remaining labels were poorly separated. |
| Qwen3.5 35B-A3B | 2/10 | Seven reasoning-only responses and one malformed JSON response. | Its apparently high macro F1 is based on only two surviving dialogues and is therefore strongly selection-biased. |
| Qwen3.5 27B | 7/10 | Three reasoning-only responses; valid outputs missed relevance and underused `A`. | More reliable than the smaller Qwen variants, but still incomplete and semantically narrow. |
| GPT-OSS 20B | 8/10 | Two responses exhausted the output allowance during reasoning; valid outputs confused `A`/`N`, missed relevance and wrong-operation errors, and invented principles/steps evidence. | Reasonable format compliance did not yield dependable annotations. |
| Gemma 3 12B | 8/10 | Two responses violated thread-resolution invariants; valid outputs dramatically overpredicted `A` and missed most misconception presence. | It produced structured output but did not apply the codebook's evidence standard. |
| Qwen3.5 9B | 0/10 | Every response used its completion budget for reasoning and produced no final JSON. | The task and full codebook exceed this configuration's practical completion capacity. |
| Gemma 3 4B | 2/10 | Eight schema failures, including combined family names, long-form labels instead of `P/A/N`, missing units, and invalid resolution links. | The model could not reliably follow the output contract; metrics from two valid dialogues are not representative. |

### Why Kimi K3 was selected

Among models with a 100% valid rate, Kimi had the strongest scores: macro F1
`0.623`, accuracy `0.926`, and Krippendorff's alpha `0.826`. Qwen3.8 Max
and GPT-OSS 120B also completed all ten dialogues, but their alphas were only
`0.455` and `0.327`. Qwen3.5 35B-A3B's higher macro F1 (`0.722`) is not a
fair counterexample because eight of its ten outputs were invalid; its score
describes only two selected dialogues.

Kimi was therefore chosen because it provided the best combination of
**output reliability**, **misconception-detection performance**, and
**overall human agreement**. This ten-dialogue screen supports choosing Kimi
for the larger validation experiment; it is not, by itself, a final estimate
of production performance.
